# 05 · Ingesta y RAG: del documento al chunk

**Objetivo:** mostrar cómo el nodo **Ingesta** indexa un documento (PDF, Markdown o texto,
sin error en los tres) y cómo el adaptador **`RecuperadorRAG`** conecta ese índice con el
agente **Investigador**.

**Qué se ve:** los 3 formatos se indexan, `recuperar()` devuelve los chunks relevantes y el
Investigador confirma cobertura usando el RAG real, no un stub.

> Sin red: usa embeddings **simulados** (bolsa de palabras). Con `HF_TOKEN` en el `.env`,
> cambia `EmbeddingsSimulados` por `EmbeddingsE5` para los puntajes reales.

In [ ]:
import sys
from pathlib import Path

# El notebook corre desde notebooks/; el código vive en la raíz del repo.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
# `src/` también entra al path: `grafo` y `llm_provider` importan de forma "pelada".
sys.path.insert(0, str(RAIZ / "src"))

import hashlib
import re
import tempfile

import numpy as np
from langchain_core.embeddings import Embeddings

from src.agentes.investigador import construir_nodo_investigador
from src.agentes.recuperador_rag import RecuperadorRAG
from src.rag.configuracion import ConfigRAG
from src.rag.indexacion import indexar_archivo
from src.rag.recuperacion import recuperar
from src.rag.vectorstore import AlmacenChroma

print("Raíz del repo:", RAIZ)

## Paso 1 · Tres documentos demo (PDF, Markdown y texto)

In [ ]:
TEXTO = (
    "El Scrum Master facilita el proceso y elimina impedimentos. "
    "El Product Owner prioriza el backlog del producto. "
) * 20

DEMO = Path(tempfile.mkdtemp(prefix="nuevamente-05-"))
(DEMO / "guia.md").write_text("# Guía de Scrum\n\n" + TEXTO, encoding="utf-8")
(DEMO / "notas.txt").write_text(TEXTO, encoding="utf-8")

from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas

lienzo = canvas.Canvas(str(DEMO / "manual.pdf"), pagesize=A4)
y = 800
for linea in TEXTO.split("."):
    lienzo.drawString(40, y, linea.strip()[:90])
    y -= 14
    if y < 60:
        lienzo.showPage(); y = 800
lienzo.save()

sorted(p.name for p in DEMO.iterdir())

## Paso 2 · Indexar los 3 formatos

`indexar_archivo()` extrae, corta en chunks, calcula embeddings y guarda en Chroma.
Un `document_id` estable permite no volver a indexar el mismo documento (D-07).

In [ ]:
class EmbeddingsSimulados(Embeddings):
    """Bolsa de palabras sin red, solo para mostrar el flujo."""

    def __init__(self, dimension: int = 768) -> None:
        self.dimension = dimension

    def _vector(self, texto: str) -> list[float]:
        v = np.zeros(self.dimension)
        for palabra in re.findall(r"\w+", texto.lower()):
            v[int(hashlib.md5(palabra.encode("utf-8")).hexdigest()[:8], 16) % self.dimension] += 1.0
        norma = np.linalg.norm(v)
        return (v / norma if norma else np.eye(1, self.dimension)[0]).tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._vector(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._vector(text)

    def embeder_consultas(self, textos: list[str]) -> list[list[float]]:
        return [self._vector(t) for t in textos]


embeddings = EmbeddingsSimulados()
almacen = AlmacenChroma(
    ConfigRAG(chroma_path=DEMO / "chroma", chunk_size=200, chunk_overlap=20, chunk_fusion=0)
)

ids: dict[str, str] = {}
for ruta in sorted(DEMO.glob("*.*")):
    if ruta.suffix == ".json":  # el índice de Chroma no es un documento
        continue
    document_id, cantidad = indexar_archivo(ruta, embeddings=embeddings, almacen=almacen)
    ids[ruta.name] = document_id
    print(f"{ruta.name:12s} -> {document_id} | {cantidad} chunk(s)")

## Paso 3 · Recuperar el fragmento que responde

In [ ]:
for chunk in recuperar(ids["guia.md"], "Scrum Master", umbral=0.0,
                      embeddings=embeddings, almacen=almacen)[:3]:
    print(f"score={chunk.score:.3f} | página={chunk.pagina} | {chunk.texto[:70]}…")

## Paso 4 · El adaptador que conecta el RAG con el Investigador

`RecuperadorRAG` implementa el protocolo `Recuperador` que espera el agente. El nodo le
pasa el `document_id` que dejó la Ingesta. El generador es un doble: si hay cobertura, el
LLM ni se usa.

In [ ]:
class GeneradorFalso:
    async def generate(self, *, prompt, output_model):
        return {"mensaje": "El documento no cubre ese tema."}


recuperador = RecuperadorRAG(embeddings=embeddings, almacen=almacen)
nodo_investigador = construir_nodo_investigador(recuperador, GeneradorFalso())

estado = {
    "tema_consulta": "Scrum Master",
    "documento_titulo": "guia.md",
    "document_id": ids["guia.md"],
}
salida = await nodo_investigador(estado)
salida

## Qué aprendimos

- La **Ingesta** cierra el ciclo: descarga → extrae → indexa → deja `document_id` en el estado.
- El **`RecuperadorRAG`** es el único cable entre el carril RAG y el agente Investigador;
  ninguno de los dos conoce al otro.
- El índice está **particionado por documento**: sin `document_id` no hay búsqueda.
- Los 3 formatos (PDF, MD, TXT) entran por la misma puerta: `rag/extraccion.py`.